# Capitolo 18 – Ricorsivo e iterativo a confronto (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap18_ricorsivo_iterativo_python.ipynb)

Gli algoritmi del capitolo, ricorsivi e iterativi, a confronto con **conteggi** (chiamate, iterazioni, confronti, profondità della ricorsione) e con qualche tempo misurato: **SumExplicitStack** (la ricorsione simulata con una pila), **MergeSort** ricorsivo e **MergeSortBottomUp**, la potenza veloce ricorsiva e **PowerIt**, **Kadane**, i tre algoritmi di Fibonacci **FibRec**, **FibMemo** e **FibIt**, **GCDRec**/**GCDIt** e la ricerca binaria ricorsiva e iterativa (eliminazione della ricorsione in coda). I conteggi sono uguali in ogni linguaggio e in ogni macchina; i tempi (in ms) dipendono dalla macchina e dal linguaggio, contano solo come crescono.

Il capitolo completo, con pseudocodice, dimostrazioni ed esempi, è [sul sito](https://fabiofurini.github.io/algoritmi/ricorsione/18-ricorsivo-iterativo/).

Ogni cella di codice si esegue con **Maiusc+Invio**; le celle vanno eseguite in ordine, dall'alto in basso. Python 3, nessuna libreria esterna.

**Indici.** Nello pseudocodice delle dispense gli array sono indicizzati da 1; nel codice (come in tutti i linguaggi qui usati) da 0: l'elemento `a[i]` dello pseudocodice è l'elemento di indice `i-1` nel codice, e un intervallo `a[l..r]` diventa `a[l-1..r-1]`. Le stampe dei notebook usano gli indici da 0.

**Nomi.** Le funzioni hanno il nome dello pseudocodice, nella convenzione del linguaggio (come nei capitoli di introduzione): `FibRec` → `fib_rec` (snake_case).

## 1. Dalla ricorsione a una pila esplicita: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma $a[1]+\dots+a[n]$. La versione iterativa della somma per dimezzamento: la pila $P$ contiene le coppie $(l,r)$ dei sotto-array ancora da sommare e fa il lavoro della pila delle chiamate. La pila non contiene mai più di $\Theta(\log_2 n)$ coppie.

Sull'array $a=[4,7,1,9]$ del capitolo si stampa la traccia (con gli indici da 0: la coppia $(1,4)$ del capitolo è qui $(0,3)$; in ogni riga la cima della pila è a destra) e si confronta con `SumIterative`, `sumRecHalving` e `sumRecDecrease` del codice del corso, con la profondità massima della ricorsione. Poi si stampa la pila delle chiamate di `sumRecDecrease(a, 3)` su $a=[4,7,1]$ (esempio del capitolo): le addizioni avvengono solo *dopo* il caso base, mentre la pila si svuota, e a un certo punto le chiamate attive sono $n+1$.

In [1]:
max_p = 0   # massima dimensione della pila P

def sum_explicit_stack(a, n):
    """SumExplicitStack(a, n): somma di a[0..n-1] con una pila esplicita."""
    global max_p
    s = 0
    P = []                          # pila dei sotto-array da sommare
    P.append((0, n - 1))
    max_p = max(max_p, len(P))
    while len(P) > 0:
        l, r = P.pop()
        if l == r:
            s = s + a[l]            # caso base
        else:
            mid = l + (r - l) // 2
            P.append((mid + 1, r))
            P.append((l, mid))
            max_p = max(max_p, len(P))
    return s

def fmt_p(P):
    return "[" + ",".join(f"({x},{y})" for x, y in P) + "]"

def sum_explicit_stack_trace(a, n):
    s = 0
    P = [(0, n - 1)]
    while len(P) > 0:
        l, r = P.pop()
        if l == r:
            s = s + a[l]
            print(f"pop ({l},{r}): caso base, s={s} => P={fmt_p(P)}")
        else:
            mid = l + (r - l) // 2
            P.append((mid + 1, r))
            P.append((l, mid))
            print(f"pop ({l},{r}): mid={mid}, push ({mid + 1},{r}) e ({l},{mid}) => P={fmt_p(P)}, s={s}")
    return s

# ---- le altre somme del codice del corso, con la profondità ----
depth = 0
max_depth = 0

def sum_it(a, n):
    s = 0
    for i in range(n):
        s = s + a[i]
    return s

def sum_rec_halving(a, l, r):
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)
    if l == r:
        result = a[l]
    else:
        mid = l + (r - l) // 2
        result = sum_rec_halving(a, l, mid) + sum_rec_halving(a, mid + 1, r)
    depth -= 1
    return result

def sum_rec_decrease(a, n):
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)
    if n == 0:
        result = 0
    else:
        result = sum_rec_decrease(a, n - 1) + a[n - 1]
    depth -= 1
    return result

a = [4, 7, 1, 9]
n = len(a)
print("a =", a, "  n =", n)
print("SumExplicitStack(a, 4):")
res = sum_explicit_stack_trace(a, n)
max_p = 0
sum_explicit_stack(a, n)
print(f"risultato {res}, dimensione massima di P = {max_p}")
print()
print("sumIt                =", sum_it(a, n))
max_depth = 0
print("sumRecHalving        =", sum_rec_halving(a, 0, n - 1), "  profondità massima =", max_depth)
max_depth = 0
print("sumRecDecrease       =", sum_rec_decrease(a, n), "  profondità massima =", max_depth)

# ---- la pila delle chiamate di sumRecDecrease(a, 3) ----
step = 0
max_active = 0

def emit(stack, action):
    global step, max_active
    step += 1
    max_active = max(max_active, len(stack))
    print(f"{step:>2d} | {' '.join(stack)} | {action}")

def sum_rec_decrease_trace(a, n, stack):
    stack.append(f"s({n})")
    if n == 0:
        emit(stack, "n=0: return 0")
        stack.pop()
        return 0
    emit(stack, f"n={n} != 0: chiama s({n - 1}), poi somma a[{n - 1}]")
    r = sum_rec_decrease_trace(a, n - 1, stack)
    emit(stack, f"return {r}+a[{n - 1}]={r + a[n - 1]}")
    stack.pop()
    return r + a[n - 1]

b = [4, 7, 1]
print()
print("a =", b, "  sumRecDecrease(a, 3)")
print(" # | pila delle chiamate attive | azione")
res = sum_rec_decrease_trace(b, 3, [])
print(f"risultato {res}, chiamate attive al massimo = {max_active} = n+1")

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) e (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) e (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): caso base, s=4 => P=[(2,3),(1,1)]
pop (1,1): caso base, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) e (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): caso base, s=12 => P=[(3,3)]
pop (3,3): caso base, s=21 => P=[]
risultato 21, dimensione massima di P = 3

sumIt                = 21
sumRecHalving        = 21   profondità massima = 3
sumRecDecrease       = 21   profondità massima = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | pila delle chiamate attive | azione
 1 | s(3) | n=3 != 0: chiama s(2), poi somma a[2]
 2 | s(3) s(2) | n=2 != 0: chiama s(1), poi somma a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: chiama s(0), poi somma a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
risultato 12, chiamate attive al massimo = 

**Prova tu.** Cambia l'array: la pila esplicita non contiene mai più di $\lceil\log_2 n\rceil+1$ coppie.

In [2]:
# Cambia l'array: la pila esplicita non contiene mai più di ceil(log2 n) + 1 coppie.
c = list(range(1, 101))
max_p = 0
print(sum_explicit_stack(c, len(c)), max_p)

5050 8


## 2. MergeSort ricorsivo e MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: un array $a[1..n]$ di naturali. *Output*: l'array ordinato in modo non decrescente. La stessa suddivisione di Merge Sort, ma **senza ricorsione**: prima si fondono coppie di sotto-array di lunghezza $1$, poi di lunghezza $2$, $4$, … con la procedura `Merge` del capitolo 17 (qui ripetuta). Esegue $\lceil\log_2 n\rceil$ passate e $\Theta(n\log_2 n)$ passi, come la versione ricorsiva, e non usa la pila delle chiamate.

Prima l'esempio del capitolo $a=[5,2,4,6,1,3]$, passata per passata (gli indici sono da 0). Poi il confronto, su array pseudo-casuali (stesso generatore in tutti i linguaggi, seme $12345$, valori fra $1$ e $10^6$), del numero di **confronti** fra elementi di `InsertionSort`, di `MergeSort` e di `MergeSortBottomUp`: $\Theta(n^2)$ contro $\Theta(n\log_2 n)$, con le due versioni di Merge Sort praticamente uguali ("OK" indica che i tre algoritmi danno lo stesso array ordinato), e i tempi misurati.

In [3]:
import time

comparisons = 0   # confronti fra elementi

def merge(a, l, m, r):
    """Merge(a, l, m, r) del capitolo 17."""
    global comparisons
    n1 = m - l + 1
    n2 = r - m
    L = [0] * n1
    R = [0] * n2
    for i in range(n1):
        L[i] = a[l + i]
    for j in range(n2):
        R[j] = a[m + 1 + j]
    i = 0
    j = 0
    k = l
    while i < n1 and j < n2:
        comparisons += 1
        if L[i] <= R[j]:
            a[k] = L[i]
            i += 1
        else:
            a[k] = R[j]
            j += 1
        k += 1
    while i < n1:
        a[k] = L[i]
        i += 1
        k += 1
    while j < n2:
        a[k] = R[j]
        j += 1
        k += 1

def merge_sort(a, l, r):
    if l < r:
        m = l + (r - l) // 2
        merge_sort(a, l, m)
        merge_sort(a, m + 1, r)
        merge(a, l, m, r)

def merge_sort_bottom_up(a, n):
    """MergeSortBottomUp(a, n): ordina a[0..n-1] senza ricorsione."""
    width = 1
    while width < n:
        l = 0
        while l + width < n:                    # pseudocodice: l + width <= n
            m = l + width - 1
            r = min(l + 2 * width - 1, n - 1)
            merge(a, l, m, r)
            l = l + 2 * width
        width = 2 * width

def merge_sort_bottom_up_trace(a, n):
    width = 1
    passes = 0
    while width < n:
        l = 0
        parts = []
        while l + width < n:
            m = l + width - 1
            r = min(l + 2 * width - 1, n - 1)
            merge(a, l, m, r)
            parts.append(f"a[{l}..{m}] con a[{m + 1}..{r}]")
            l = l + 2 * width
        passes += 1
        print(f"width={width}: merge " + ", ".join(parts) + f" => {a}")
        width = 2 * width
    print(f"passate = {passes}")

def insertion_sort(a, n):
    """InsertionSort(a, n) (capitolo 3), con il conteggio dei confronti."""
    global comparisons
    for j in range(1, n):
        key = a[j]
        i = j - 1
        while i >= 0:
            comparisons += 1
            if a[i] > key:
                a[i + 1] = a[i]
                i = i - 1
            else:
                break
        a[i + 1] = key

def random_array(n, seed=12345):
    """Generatore congruenziale lineare: stessi numeri in Python, C++ e Java."""
    x = seed
    a = []
    for _ in range(n):
        x = (x * 1103515245 + 12345) % 2147483648
        a.append(1 + x % 1000000)
    return a

a = [5, 2, 4, 6, 1, 3]
print("a =", a, "  n =", len(a))
merge_sort_bottom_up_trace(a, len(a))

print()
print(f"{'n':>6s} {'InsertionSort':>14s} {'MergeSort':>10s} {'BottomUp':>9s}  confronti")
times = []
for n in [1000, 2000, 4000]:
    base = random_array(n)
    x1, x2, x3 = base[:], base[:], base[:]
    comparisons = 0
    t0 = time.perf_counter(); insertion_sort(x1, n); t1 = (time.perf_counter() - t0) * 1000
    c1 = comparisons
    comparisons = 0
    t0 = time.perf_counter(); merge_sort(x2, 0, n - 1); t2 = (time.perf_counter() - t0) * 1000
    c2 = comparisons
    comparisons = 0
    t0 = time.perf_counter(); merge_sort_bottom_up(x3, n); t3 = (time.perf_counter() - t0) * 1000
    c3 = comparisons
    ok = "OK" if (x1 == x2 == x3 == sorted(base)) else "ERROR"
    print(f"{n:>6d} {c1:>14d} {c2:>10d} {c3:>9d}  {ok}")
    times.append((n, t1, t2, t3))
print()
print("Tempi misurati (dipendono dalla macchina):")
for n, t1, t2, t3 in times:
    print(f"n={n:<6d} InsertionSort {t1:>10.3f} ms   MergeSort {t2:>8.3f} ms   MergeSortBottomUp {t3:>8.3f} ms")

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] con a[1..1], a[2..2] con a[3..3], a[4..4] con a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] con a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] con a[4..5] => [1, 2, 3, 4, 5, 6]
passate = 3

     n  InsertionSort  MergeSort  BottomUp  confronti
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Tempi misurati (dipendono dalla macchina):
n=1000   InsertionSort     13.973 ms   MergeSort    1.122 ms   MergeSortBottomUp    1.105 ms
n=2000   InsertionSort     46.387 ms   MergeSort    1.964 ms   MergeSortBottomUp    1.924 ms
n=4000   InsertionSort    164.047 ms   MergeSort    4.193 ms   MergeSortBottomUp    4.183 ms


**Prova tu.** Cambia $n$ (con $n=8000$ i confronti di `InsertionSort` quadruplicano rispetto a $n=4000$, quelli di Merge Sort poco più del doppio).

In [4]:
# Prova con n = 8000: i confronti di InsertionSort quadruplicano rispetto a n = 4000, quelli di Merge Sort poco più del doppio.
n = 8000
base = random_array(n)
x1, x2 = base[:], base[:]
comparisons = 0
insertion_sort(x1, n)
c1 = comparisons
comparisons = 0
merge_sort_bottom_up(x2, n)
print(c1, comparisons, x1 == x2)

16115652 93756 True


## 3. Potenza veloce: ricorsiva e PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — ricorsiva: $x^n=(x^{\lfloor n/2\rfloor})^2$ se $n$ è pari, $(x^{\lfloor n/2\rfloor})^2\cdot x$ se è dispari. $\Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor+2$ chiamate (profondità $\Theta(\log_2 n)$). Il risultato della chiamata ricorsiva è salvato in $y$ e usato due volte.
- **PowerIt**$(x,n)$ — *Input*: un numero $x$ e un naturale $n$. *Output*: $x^n$. Legge i bit di $n$ dal meno significativo: $\lfloor\log_2 n\rfloor+1$ iterazioni, memoria extra $\Theta(1)$; vale $p\cdot y^k=x^n$ a ogni iterazione.

Con $x=2$ e $n=13=(1101)_2$ si stampa la traccia del capitolo: $p$ viene moltiplicato per $x$, $x^4$ e $x^8$ (i bit di 13 uguali a 1) e $p=x^{13}$ dopo $4$ iterazioni. Poi la tabella: stesso risultato, $\lfloor\log_2 n\rfloor+1$ iterazioni contro $\lfloor\log_2 n\rfloor+2$ chiamate, contro le $n$ moltiplicazioni del ciclo naïf.

In [5]:
pow_calls = 0   # chiamate di FastPower
pow_iters = 0   # iterazioni di PowerIt

def fast_power(x, n):
    """FastPower(x, n): x^n per ricorsione sul dimezzamento di n."""
    global pow_calls
    pow_calls += 1
    if n == 0:
        return 1                    # caso base
    y = fast_power(x, n // 2)       # una sola chiamata ricorsiva
    if n % 2 == 0:
        return y * y                # n pari
    return x * y * y                # n dispari

def power_it(x, n):
    """PowerIt(x, n): x^n con square-and-multiply."""
    global pow_iters
    p = 1
    y = x
    k = n
    while k > 0:
        pow_iters += 1
        if k % 2 == 1:
            p = p * y
        y = y * y
        k = k // 2
    return p

def power_it_trace(x, n):
    print(f"PowerIt({x}, {n}): {n} = ({bin(n)[2:]})_2")
    p = 1
    y = x
    k = n
    it = 0
    while k > 0:
        it += 1
        par = "dispari" if k % 2 == 1 else "pari"
        if k % 2 == 1:
            p = p * y
        y = y * y
        print(f"{it}. k={k} {par}: p={p}, y={y}, k={k // 2}")
        k = k // 2
    print(f"iterazioni = {it}, risultato p = {p}")

power_it_trace(2, 13)
print()
print(f"{'n':>3s} {'2^n':>12s} {'PowerIt':>8s} {'FastPower':>10s} {'naive':>6s}   (iterazioni, chiamate, moltiplicazioni)")
for n in [0, 1, 2, 5, 13, 30]:
    pow_iters = 0
    r1 = power_it(2, n)
    pow_calls = 0
    r2 = fast_power(2, n)
    print(f"{n:>3d} {r1:>12d} {pow_iters:>8d} {pow_calls:>10d} {n:>6d}   {'OK' if r1 == r2 == 2 ** n else 'ERROR'}")

PowerIt(2, 13): 13 = (1101)_2
1. k=13 dispari: p=2, y=4, k=6
2. k=6 pari: p=2, y=16, k=3
3. k=3 dispari: p=32, y=256, k=1
4. k=1 dispari: p=8192, y=65536, k=0
iterazioni = 4, risultato p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterazioni, chiamate, moltiplicazioni)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Prova tu.** Prova un'altra base e un altro esponente (in C++ e Java il risultato deve stare in 64 bit).

In [6]:
# Prova un'altra base e un altro esponente (attenzione: in C++ e Java il risultato deve stare in 64 bit).
pow_iters = 0
pow_calls = 0
print(power_it(3, 25), pow_iters, fast_power(3, 25), pow_calls)

847288609443 5 847288609443 6


## 4. Kadane: un algoritmo iterativo migliore

**Kadane**$(a,n)$ — *Input*: un array $a[1..n]$ di $n\ge1$ interi. *Output*: la somma massima di un sottoarray non vuoto $a[l..r]$. È la versione compatta del capitolo: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. Il ciclo fa $n-1$ iterazioni: $\Theta(n)$ passi e $\Theta(1)$ memoria, contro le $n(n+1)/2$ coppie $(l,r)$ della forza bruta ($\Theta(n^2)$) e contro il divide et impera ricorsivo ($\Theta(n\log_2 n)$): un buon algoritmo iterativo può battere la ricorsione.

Sull'esempio delle variazioni di prezzo del capitolo 17 il risultato è $43$. Poi, su array pseudo-casuali con valori fra $-50$ e $50$ (stesso generatore in tutti i linguaggi, seme $n$), le iterazioni di `Kadane`, le coppie esaminate dalla forza bruta e la verifica che i due algoritmi trovino la stessa somma massima; infine i tempi misurati.

In [7]:
import time

def kadane(a, n):
    """Kadane(a, n): somma massima di un sottoarray non vuoto di a[0..n-1]."""
    best = a[0]
    cur = a[0]                      # somma massima di un sottoarray che termina in i
    i = 1
    while i < n:
        cur = max(a[i], cur + a[i])
        best = max(best, cur)
        i = i + 1
    return best

def brute_force_best(a, n):
    """Forza bruta: somma massima e numero di coppie (l, r) esaminate."""
    best = a[0]
    pairs = 0
    for l in range(n):
        s = 0
        for r in range(l, n):
            s = s + a[r]
            pairs += 1
            if s > best:
                best = s
    return best, pairs

def random_values(n, seed=12345):
    """Interi fra -50 e 50 (generatore congruenziale lineare)."""
    x = seed
    a = []
    for _ in range(n):
        x = (x * 1103515245 + 12345) % 2147483648
        a.append(x % 101 - 50)
    return a

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
print("a =", a)
print("Kadane(a, 16) =", kadane(a, len(a)))
b = [-3, -1, -2, -5]
print("a =", b, " Kadane(a, 4) =", kadane(b, len(b)))
print()
print(f"{'n':>6s} {'Kadane':>7s} {'BruteForce':>11s} {'iterations':>11s} {'pairs':>9s}  stessa somma?")
times = []
for n in [1000, 2000, 4000]:
    v = random_values(n, n)
    t0 = time.perf_counter(); k = kadane(v, n); t1 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); bf, pairs = brute_force_best(v, n); t2 = (time.perf_counter() - t0) * 1000
    print(f"{n:>6d} {k:>7d} {bf:>11d} {n - 1:>11d} {pairs:>9d}  {'OK' if k == bf else 'ERROR'}")
    times.append((n, t1, t2))
print()
print("Tempi misurati (dipendono dalla macchina):")
for n, t1, t2 in times:
    print(f"n={n:<6d} Kadane {t1:>9.4f} ms   BruteForce {t2:>10.3f} ms")

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  stessa somma?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Tempi misurati (dipendono dalla macchina):
n=1000   Kadane    0.0579 ms   BruteForce     14.716 ms
n=2000   Kadane    0.1135 ms   BruteForce     60.188 ms
n=4000   Kadane    0.2193 ms   BruteForce    239.730 ms


**Prova tu.** Cambia $n$: le coppie della forza bruta crescono come $n^2$, le iterazioni di Kadane come $n$.

In [8]:
# Prova con n = 8000: le coppie della forza bruta sono n(n+1)/2 = 32 004 000, le iterazioni di Kadane 7999.
n = 8000
v = random_values(n)
print(kadane(v, n), brute_force_best(v, n))

1903 (1903, 32004000)


## 5. Fibonacci: FibRec, FibMemo e FibIt (sottoproblemi che si sovrappongono)

*Input*: un naturale $n$. *Output*: il numero di Fibonacci $F(n)$, con $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — traduzione diretta della definizione: i sottoproblemi si sovrappongono e le chiamate sono $2F(n+1)-1=\Theta(\varphi^n)$, esponenziali; la profondità è $n$.
- **FibMemo**$(n,memo)$ — ricorsione con memoizzazione: ogni valore è calcolato una volta (il vettore `memo` vale $-1$ per «non definito»); $2n-1$ chiamate, memoria $\Theta(n)$.
- **FibIt**$(n)$ — ciclo con due variabili: $n-1$ iterazioni, memoria $\Theta(1)$.

Sull'esempio $\texttt{FibRec}(5)$ del capitolo si conta quante volte è chiamata ogni sottoproblema (l'albero ha $15$ nodi, ma servono solo $6$ valori diversi). Poi la tabella dei conteggi (che coincidono con i valori del capitolo: $21\,891$, $242\,785$, $2\,692\,537$ chiamate per $n=20,25,30$) e i tempi misurati.

In [9]:
import time

fib_calls = 0   # chiamate eseguite

def fib_rec(n):
    """FibRec(n)"""
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    return fib_rec(n - 1) + fib_rec(n - 2)

def fib_memo(n, memo):
    """FibMemo(n, memo): memo[k] = -1 significa «non definito»."""
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    if memo[n] == -1:
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

def fib_it(n):
    """FibIt(n)"""
    if n <= 1:
        return n
    prev = 0                        # F(i-2)
    curr = 1                        # F(i-1)
    i = 2
    while i <= n:
        nxt = prev + curr
        prev = curr
        curr = nxt
        i = i + 1
    return curr

def fib_rec_hist(n, counts):
    """Come FibRec, ma conta le chiamate per ogni valore del parametro."""
    counts[n] += 1
    if n <= 1:
        return n
    return fib_rec_hist(n - 1, counts) + fib_rec_hist(n - 2, counts)

counts = [0] * 6
print("FibRec(5) =", fib_rec_hist(5, counts))
print("chiamate per ogni valore del parametro:", "  ".join(f"{k}:{counts[k]}" for k in range(5, -1, -1)), f"  totale {sum(counts)}")
print()
print(f"{'n':>3s} {'F(n)':>8s} {'FibRec':>9s} {'2F(n+1)-1':>10s} {'FibMemo':>8s} {'FibIt':>6s}  (chiamate, chiamate, chiamate, iterazioni)")
for n in [10, 15, 20, 25, 30]:
    f = fib_it(n)
    fib_calls = 0
    r1 = fib_rec(n)
    c1 = fib_calls
    fib_calls = 0
    r2 = fib_memo(n, [-1] * (n + 1))
    c2 = fib_calls
    ok = "OK" if (r1 == r2 == f and c1 == 2 * fib_it(n + 1) - 1 and c2 == 2 * n - 1) else "ERROR"
    print(f"{n:>3d} {f:>8d} {c1:>9d} {2 * fib_it(n + 1) - 1:>10d} {c2:>8d} {n - 1:>6d}  {ok}")
print()
print("Tempi misurati (dipendono dalla macchina):")
for n in [20, 25, 30]:
    t0 = time.perf_counter(); fib_rec(n); t1 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter()
    for _ in range(1000):
        fib_it(n)
    t2 = (time.perf_counter() - t0) * 1000 / 1000
    print(f"n={n:<3d} FibRec {t1:>10.4f} ms   FibIt {t2:>10.6f} ms")

FibRec(5) = 5
chiamate per ogni valore del parametro: 5:1  4:1  3:2  2:3  1:5  0:3   totale 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (chiamate, chiamate, chiamate, iterazioni)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Tempi misurati (dipendono dalla macchina):
n=20  FibRec     0.7628 ms   FibIt   0.000385 ms
n=25  FibRec     8.0596 ms   FibIt   0.000506 ms
n=30  FibRec    89.4816 ms   FibIt   0.000566 ms


**Prova tu.** Aumenta $n$ (con $n=32$ `FibRec` esegue $7\,049\,155$ chiamate, `FibMemo` $63$) e guarda come cresce il tempo di `FibRec`.

In [10]:
# Aumenta n: con n = 32 FibRec esegue 7 049 155 chiamate, FibMemo 63.
n = 32
fib_calls = 0
print(fib_rec(n), fib_calls)
fib_calls = 0
print(fib_memo(n, [-1] * (n + 1)), fib_calls)

2178309 7049155
2178309 63


## 6. Ricorsione in coda e la sua eliminazione: GCDRec/GCDIt, RecBinarySearch/ItBinarySearch

Una chiamata ricorsiva è *in coda* se è l'ultima operazione e il suo risultato è restituito così com'è: si può sostituire con l'aggiornamento dei parametri e un salto all'inizio, cioè con un ciclo (condizione del ciclo = negazione del caso base; chiamata in coda = assegnamento simultaneo dei parametri). Il risultato è lo stesso, il numero di passi ha lo stesso ordine di grandezza e la memoria extra scende da $\Theta(\text{profondità})$ a $\Theta(1)$.

- **GCDRec**$(a,b)$ / **GCDIt**$(a,b)$ — *Input*: due naturali $a$, $b$. *Output*: il loro massimo comun divisore (algoritmo di Euclide). Con $(a,b)=(252,105)$ entrambi passano per le coppie $(252,105)$, $(105,42)$, $(42,21)$, $(21,0)$ e restituiscono $21$: quattro chiamate contro tre iterazioni. La variabile temporanea $r$ serve perché $a\bmod b$ va calcolato con il *vecchio* $a$.
- **RecBS**$(a,l,r,m)$ / **ItBS**$(a,l,r,m)$ — *Input*: un array ordinato, due indici $l$, $r$ e un valore $m$. *Output*: un indice di $m$ in $a[l..r]$ oppure $-1$. `ItBS(a, 1, n, m)` è la ricerca binaria iterativa del capitolo «Ricerca».

In [11]:
gcd_log = []    # coppie (a, b) visitate (solo per la stampa)
rbs_calls = 0   # chiamate di RecBS
bs_iters = 0    # iterazioni di ItBS

def gcd_rec(a, b):
    """GCDRec(a, b)"""
    gcd_log.append(f"({a},{b})")
    if b == 0:
        return a
    return gcd_rec(b, a % b)            # chiamata in coda

def gcd_it(a, b):
    """GCDIt(a, b)"""
    gcd_log.append(f"({a},{b})")
    while b != 0:                       # negazione del caso base
        r = a % b                       # con il vecchio a
        a = b
        b = r
        gcd_log.append(f"({a},{b})")
    return a

def rec_bs(a, l, r, m):
    """RecBS(a, l, r, m)"""
    global rbs_calls
    rbs_calls += 1
    if l > r:
        return -1
    mid = l + (r - l) // 2
    if a[mid] == m:
        return mid
    if a[mid] < m:
        return rec_bs(a, mid + 1, r, m)     # chiamata in coda
    else:
        return rec_bs(a, l, mid - 1, m)     # chiamata in coda

def it_bs(a, l, r, m):
    """ItBS(a, l, r, m)"""
    global bs_iters
    while l <= r:                           # negazione del caso base l > r
        bs_iters += 1
        mid = l + (r - l) // 2
        if a[mid] == m:
            return mid                      # il secondo caso base resta nel ciclo
        if a[mid] < m:
            l = mid + 1
        else:
            r = mid - 1
    return -1

for a, b in [(252, 105), (1071, 462), (832040, 514229)]:
    gcd_log.clear()
    g1 = gcd_rec(a, b)
    calls = len(gcd_log)
    pairs = " ".join(gcd_log)
    gcd_log.clear()
    g2 = gcd_it(a, b)
    iters = len(gcd_log) - 1
    print(f"GCDRec({a}, {b}) = {g1}   chiamate={calls}     GCDIt = {g2}   iterazioni={iters}   {'OK' if g1 == g2 else 'ERROR'}")
    if a == 252:
        print("  coppie:", pairs)

print()
a = [3, 9, 11, 12, 12, 16, 19]
n = len(a)
print("a =", a, "  n =", n)
print(f"{'m':>4s} {'RecBS':>6s} {'calls':>6s} {'ItBS':>6s} {'iterations':>11s}")
for m in [12, 10, 3, 19, 100]:
    rbs_calls = 0
    r1 = rec_bs(a, 0, n - 1, m)
    bs_iters = 0
    r2 = it_bs(a, 0, n - 1, m)
    print(f"{m:>4d} {r1:>6d} {rbs_calls:>6d} {r2:>6d} {bs_iters:>11d}   {'OK' if r1 == r2 else 'ERROR'}")

GCDRec(252, 105) = 21   chiamate=4     GCDIt = 21   iterazioni=3   OK
  coppie: (252,105) (105,42) (42,21) (21,0)
GCDRec(1071, 462) = 21   chiamate=4     GCDIt = 21   iterazioni=3   OK
GCDRec(832040, 514229) = 1   chiamate=29     GCDIt = 1   iterazioni=28   OK

a = [3, 9, 11, 12, 12, 16, 19]   n = 7
   m  RecBS  calls   ItBS  iterations
  12      3      1      3           1   OK
  10     -1      4     -1           3   OK
   3      0      3      0           3   OK
  19      6      3      6           3   OK
 100     -1      4     -1           3   OK


**Prova tu.** Prova altre coppie per il GCD: con due numeri di Fibonacci consecutivi l'algoritmo di Euclide fa il maggior numero di passi.

In [12]:
# Prova altre coppie per il GCD: con due numeri di Fibonacci consecutivi l'algoritmo di Euclide fa il maggior numero di passi.
gcd_log.clear()
print(gcd_rec(46368, 28657), len(gcd_log))

1 23


## 7. Profondità della ricorsione e pila delle chiamate

La memoria usata dalla pila delle chiamate è proporzionale alla **profondità** $D(n)$ della ricorsione (il massimo numero di chiamate attive contemporaneamente), non al numero di chiamate. La tabella confronta la profondità **misurata** con la formula del capitolo:

| algoritmo | $D(n)$ |
|---|---|
| somma per decremento di uno (caso base $n=0$) | $n+1$ |
| somma per dimezzamento | $\lceil\log_2 n\rceil+1$ |
| FastPower | $\lfloor\log_2 n\rfloor+2$ |
| RecBS (caso peggiore, valore assente) | $\le\lfloor\log_2 n\rfloor+2$ |
| FibRec$(n)$, $n\ge1$ | $n$ |

Una profondità $\Theta(\log_2 n)$ non è mai un problema in pratica; una profondità $\Theta(n)$ rischia lo **stack overflow**: l'ultima parte della cella (le righe che iniziano con `~~`) dipende dal linguaggio e dalla macchina.

In [13]:
import sys

depth = 0
max_depth = 0
calls = 0

def enter():
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)

def sum_rec_decrease(a, n):
    enter()
    result = 0 if n == 0 else sum_rec_decrease(a, n - 1) + a[n - 1]
    exit_call()
    return result

def sum_rec_halving(a, l, r):
    enter()
    if l == r:
        result = a[l]
    else:
        mid = l + (r - l) // 2
        result = sum_rec_halving(a, l, mid) + sum_rec_halving(a, mid + 1, r)
    exit_call()
    return result

def fast_power(x, n):
    enter()
    if n == 0:
        result = 1
    else:
        y = fast_power(x, n // 2)
        result = y * y if n % 2 == 0 else x * y * y
    exit_call()
    return result

def rec_bs(a, l, r, m):
    global calls
    calls += 1
    if l > r:
        return -1
    mid = l + (r - l) // 2
    if a[mid] == m:
        return mid
    if a[mid] < m:
        return rec_bs(a, mid + 1, r, m)
    return rec_bs(a, l, mid - 1, m)

def fib_rec(n):
    enter()
    result = n if n <= 1 else fib_rec(n - 1) + fib_rec(n - 2)
    exit_call()
    return result

def exit_call():
    global depth
    depth -= 1

def floor_log2(n):
    return n.bit_length() - 1

def ceil_log2(n):
    return (n - 1).bit_length()

print(f"{'n':>5s} {'decrease':>9s} {'n+1':>5s} {'halving':>8s} {'formula':>8s} {'FastPower':>10s} {'formula':>8s} {'RecBS':>6s} {'bound':>6s}")
for n in [10, 100, 500]:
    a = list(range(n))
    max_depth = 0; sum_rec_decrease(a, n); d1 = max_depth
    max_depth = 0; sum_rec_halving(a, 0, n - 1); d2 = max_depth
    max_depth = 0; fast_power(1, n); d3 = max_depth
    calls = 0; rec_bs(a, 0, n - 1, n + 100); d4 = calls
    print(f"{n:>5d} {d1:>9d} {n + 1:>5d} {d2:>8d} {ceil_log2(n) + 1:>8d} {d3:>10d} {floor_log2(n) + 2:>8d} {d4:>6d} {floor_log2(n) + 2:>6d}")
print()
print(f"{'n':>3s} {'FibRec depth':>13s}")
for n in [5, 10, 20]:
    max_depth = 0; fib_rec(n)
    print(f"{n:>3d} {max_depth:>13d}")

print(f"~~ Python: recursion limit = {sys.getrecursionlimit()}")
try:
    depth = 0
    sum_rec_decrease([0] * 100000, 100000)
    print("~~ Python: n = 100000 OK")
except RecursionError:
    depth = 0
    print("~~ Python: RecursionError for sum_rec_decrease with n = 100000 (depth n+1 > limit)")

    n  decrease   n+1  halving  formula  FastPower  formula  RecBS  bound
   10        11    11        5        5          5        5      5      5
  100       101   101        8        8          8        8      8      8
  500       501   501       10       10         10       10     10     10

  n  FibRec depth
  5             5
 10            10
 20            20
~~ Python: recursion limit = 1000
~~ Python: RecursionError for sum_rec_decrease with n = 100000 (depth n+1 > limit)


**Prova tu.** Misura la profondità per un altro $n$ (qui $2000$): `decrease` $=n+1$, `halving` $=\lceil\log_2 n\rceil+1$.

In [14]:
# Misura la profondità della ricorsione per un altro n (qui 2000): decrease = n+1, halving = ceil(log2 n) + 1.
sys.setrecursionlimit(5000)         # serve una profondità di 2001 chiamate
n = 2000
a = [1] * n
depth = 0
max_depth = 0
sum_rec_decrease(a, n)
print(max_depth, end=" ")
max_depth = 0
sum_rec_halving(a, 0, n - 1)
print(max_depth)
sys.setrecursionlimit(1000)

2001 12
